# Viterbi Algorithm for HMM Genome Segmentation

**Author:** Zhuang_zhanquan  
**Student ID:** 60209978
**Course:** EE8403  
**Assignment:** Homework 1

## Task Overview

Implement a Viterbi parser for a two-state Hidden Markov Model (HMM) to segment DNA sequences into regions of different GC composition.

- **State A**: AT-biased (emission probabilities: A=0.35, C=0.15, G=0.15, T=0.35)
- **State B**: GC-biased (emission probabilities: A=0.15, C=0.35, G=0.35, T=0.15)
- **Transition probabilities**: A→B=0.001, B→A=0.01
- **Initial distribution**: A=0.5, B=0.5

## Part 1: Reading Input Files

In [16]:
import sys
import math

def read_hmm_file(hmm_path):
    """
    Read HMM parameters from file.
    Format:
        Line 1: num_states num_symbols alphabet (e.g., "2 4 ACGT")
        Line 2: initial probabilities for each state
        Lines 3+: transition probabilities + emission probabilities for each state
    """
    with open(hmm_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip() and not line.strip().startswith('//')]

    # Parse first line: number of states, symbols, and alphabet
    first_line = lines[0].split()
    num_states = int(first_line[0])
    num_symbols = int(first_line[1])
    alphabet = list(first_line[2])  # e.g., ['A', 'C', 'G', 'T']

    # State names: state A, state B, ...
    states = [f"state {chr(65+i)}" for i in range(num_states)]

    # Parse initial probabilities
    initial_probs_list = [float(x) for x in lines[1].split()]
    initial_probs = {states[i]: initial_probs_list[i] for i in range(num_states)}

    # Parse transition and emission probabilities for each state
    transition_probs = {}
    emission_probs = {}

    for i in range(num_states):
        line_parts = lines[2 + i].split()
        trans_probs = [float(x) for x in line_parts[:num_states]]
        emit_probs = [float(x) for x in line_parts[num_states:num_states + num_symbols]]

        for j in range(num_states):
            transition_probs[(states[i], states[j])] = trans_probs[j]

        for j, symbol in enumerate(alphabet):
            emission_probs[(states[i], symbol)] = emit_probs[j]

    return states, alphabet, initial_probs, transition_probs, emission_probs


def read_fasta_file(fasta_path):
    """
    Read DNA sequence from FASTA file.
    Handles multi-line sequences. Converts to uppercase.
    """
    sequence = []
    with open(fasta_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                continue
            sequence.append(line.upper())

    return ''.join(sequence)

## Part 2: Viterbi Algorithm Implementation

The Viterbi algorithm uses dynamic programming（DP） to find the most likely sequence of hidden states.

**Key steps:**
1. **Table Generation**: Create Viterbi and backpointer tables
2. **Table Initialization**: Set initial probabilities for position 0
3. **Table Filling**: Dynamic programming to fill the Viterbi table
4. **Traceback**: Find the best path by backtracking through the backpointers

**Important**: Use log probabilities to avoid numerical underflow (sequence length can be 1 million+).

In [17]:
def viterbi_algorithm(sequence, states, alphabet, initial_probs, transition_probs, emission_probs):
    """
    Viterbi algorithm using log probabilities to avoid underflow.
    """
    n = len(sequence)
    num_states = len(states)

    # --- Table Generation ---
    # viterbi[i][j] = max log probability of being in state j at position i
    viterbi = [[float('-inf')] * num_states for _ in range(n)]
    
    # backptr[i][j] = previous state that led to max probability
    backptr = [[0] * num_states for _ in range(n)]

    # --- Table Initialization (position 0) ---
    first_symbol = sequence[0]
    for j, state in enumerate(states):
        init_prob = initial_probs[state]
        emit_prob = emission_probs[(state, first_symbol)]
        if init_prob > 0 and emit_prob > 0:
            viterbi[0][j] = math.log(init_prob) + math.log(emit_prob)
        else:
            viterbi[0][j] = float('-inf')

    # Precompute log values for speed
    log_trans = {}
    for (s_from, s_to), p in transition_probs.items():
        log_trans[(s_from, s_to)] = math.log(p) if p > 0 else float('-inf')

    log_emit = {}
    for (s, sym), p in emission_probs.items():
        log_emit[(s, sym)] = math.log(p) if p > 0 else float('-inf')

    # --- Table Filling (positions 1..n-1) ---
    for i in range(1, n):
        symbol = sequence[i]
        for j, cur_state in enumerate(states):
            max_prob = float('-inf')
            best_prev = 0
            le = log_emit[(cur_state, symbol)]
            for k, prev_state in enumerate(states):
                prob = viterbi[i-1][k] + log_trans[(prev_state, cur_state)] + le
                if prob > max_prob:
                    max_prob = prob
                    best_prev = k
            viterbi[i][j] = max_prob
            backptr[i][j] = best_prev

    # --- Traceback ---
    best_path = [0] * n
    best_path[n-1] = max(range(num_states), key=lambda j: viterbi[n-1][j])

    for i in range(n-2, -1, -1):
        best_path[i] = backptr[i+1][best_path[i+1]]

    return [states[idx] for idx in best_path]

## Part 3: Output Formatting

In [18]:
def format_output(path_states):
    """
    Group consecutive same-state positions into segments.
    Positions are 1-indexed in output.
    """
    if not path_states:
        return [], 0

    segments = []
    current_state = path_states[0]
    current_start = 0

    for i in range(1, len(path_states)):
        if path_states[i] != current_state:
            segments.append((current_start + 1, i, current_state))
            current_state = path_states[i]
            current_start = i

    segments.append((current_start + 1, len(path_states), current_state))

    state_b_count = sum(1 for _, _, state in segments if state == "state B")
    return segments, state_b_count


def run_viterbi(hmm_file, fasta_file):
    """Main function to run Viterbi algorithm."""
    # Read input files
    states, alphabet, initial_probs, transition_probs, emission_probs = read_hmm_file(hmm_file)
    sequence = read_fasta_file(fasta_file)

    # Run Viterbi algorithm
    path_states = viterbi_algorithm(sequence, states, alphabet, initial_probs, transition_probs, emission_probs)

    # Format and print output
    segments, state_b_count = format_output(path_states)
    
    print("Segment Output:")
    for start, end, state in segments:
        print(f"{start} {end} {state}")

    print(f"\nNumber of segments in state B: {state_b_count}")
    return segments, state_b_count

## Part 4: Testing on example.fa (1 Mb sequence)

In [19]:
# Run on example.fa
segments, state_b_count = run_viterbi('example.hmm', 'example.fa')

print(f"\nTotal segments: {len(segments)}")
print(f"State B segments: {state_b_count}")

Segment Output:
1 52 state B
53 720 state A
721 768 state B
769 2257 state A
2258 2404 state B
2405 3472 state A
3473 3526 state B
3527 5443 state A
5444 5516 state B
5517 6331 state A
6332 6369 state B
6370 6702 state A
6703 6929 state B
6930 7140 state A
7141 7176 state B
7177 8408 state A
8409 8444 state B
8445 10348 state A
10349 10408 state B
10409 11371 state A
11372 11584 state B
11585 12342 state A
12343 12439 state B
12440 12690 state A
12691 12914 state B
12915 14260 state A
14261 14308 state B
14309 18224 state A
18225 18349 state B
18350 18444 state A
18445 18687 state B
18688 19773 state A
19774 19805 state B
19806 21211 state A
21212 21253 state B
21254 21725 state A
21726 21835 state B
21836 22338 state A
22339 22358 state B
22359 22872 state A
22873 22922 state B
22923 24259 state A
24260 24598 state B
24599 25137 state A
25138 25178 state B
25179 25534 state A
25535 25588 state B
25589 27139 state A
27140 27432 state B
27433 27574 state A
27575 27647 state B
27648 2788

## Part 4b: Verification Against Ground Truth

Compare our Viterbi output against the ground truth in `example.positions`.
The `.positions` file contains the true state labels (derived from the simulation: lowercase = state A, uppercase = state B).

We compare at two levels:
1. **Per-position accuracy**: what fraction of the 1 Mb positions match
2. **Segment-level comparison**: how many ground-truth segments we recover correctly

In [20]:
def parse_positions_file(path):
    """Parse a .positions file into a list of (start, end, state) tuples."""
    segments = []
    with open(path, 'r') as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 4:
                start, end = int(parts[0]), int(parts[1])
                state = f"{parts[2]} {parts[3]}"  # 'state A' or 'state B'
                segments.append((start, end, state))
    return segments


def positions_to_array(segments, length):
    """Convert segment list to per-position state array (1-indexed)."""
    arr = [None] * (length + 1)
    for start, end, state in segments:
        for p in range(start, end + 1):
            arr[p] = state
    return arr


# --- Load ground truth ---
gt_segments = parse_positions_file('example.positions')
seq_len = max(s[1] for s in gt_segments)
gt_arr = positions_to_array(gt_segments, seq_len)

# --- Load our prediction (from cell above) ---
my_arr = positions_to_array(segments, seq_len)

# --- Per-position accuracy ---
agree = sum(1 for p in range(1, seq_len + 1) if my_arr[p] == gt_arr[p])
total = seq_len
accuracy = agree / total * 100

print("=" * 60)
print("VERIFICATION: Viterbi Output vs Ground Truth")
print("=" * 60)
print(f"Sequence length: {total:,} bp")
print(f"Positions matching: {agree:,} / {total:,}")
print(f"Per-position accuracy: {accuracy:.2f}%")
print()

# --- Segment-level comparison ---
print(f"Our segments: {len(segments)}")
print(f"GT segments:  {len(gt_segments)}")
print(f"Our state B count: {sum(1 for s in segments if s[2] == 'state B')}")
print(f"GT state B count:  {sum(1 for s in gt_segments if s[2] == 'state B')}")
print()

# --- Show first N segments side-by-side ---
print("First 15 segments comparison (ours vs ground truth):")
print(f"{'#':>3}  {'Our start':>10} {'Our end':>10} {'State':>8}  |  {'GT start':>10} {'GT end':>10} {'State':>8}")
print("-" * 75)
n_show = min(15, max(len(segments), len(gt_segments)))
for i in range(n_show):
    if i < len(segments):
        ms, me, mst = segments[i]
        our_str = f"{ms:>10} {me:>10} {mst:>8}"
    else:
        our_str = "         -          -        -"
    if i < len(gt_segments):
        gs, ge, gst = gt_segments[i]
        gt_str = f"{gs:>10} {ge:>10} {gst:>8}"
    else:
        gt_str = "         -          -        -"
    # Mark match/mismatch
    if i < len(segments) and i < len(gt_segments):
        match = "✓" if (ms == gs and me == ge and mst == gst) else "≈" if mst == gst else "✗"
    else:
        match = " "
    print(f"{i+1:>3}  {our_str}  |  {gt_str}  {match}")

print()
print(f"Summary: {accuracy:.2f}% per-position accuracy across {total:,} bp.")

VERIFICATION: Viterbi Output vs Ground Truth
Sequence length: 1,000,000 bp
Positions matching: 990,367 / 1,000,000
Per-position accuracy: 99.04%

Our segments: 1376
GT segments:  1872
Our state B count: 688
GT state B count:  936

First 15 segments comparison (ours vs ground truth):
  #   Our start    Our end    State  |    GT start     GT end    State
---------------------------------------------------------------------------
  1           1         52  state B  |           1         52  state B  ✓
  2          53        720  state A  |          53        720  state A  ✓
  3         721        768  state B  |         721        770  state B  ≈
  4         769       2257  state A  |         771       2255  state A  ≈
  5        2258       2404  state B  |        2256       2415  state B  ≈
  6        2405       3472  state A  |        2416       2983  state A  ≈
  7        3473       3526  state B  |        2984       3002  state B  ≈
  8        3527       5443  state A  |        3003 

## Part 5: Testing on Additional Files

In [21]:
# Test on test1.fa
print("=== test1.fa ===")
run_viterbi('example.hmm', 'test1.fa')

=== test1.fa ===
Segment Output:
1 15 state A
16 64 state B
65 129 state A
130 151 state B

Number of segments in state B: 2


([(1, 15, 'state A'),
  (16, 64, 'state B'),
  (65, 129, 'state A'),
  (130, 151, 'state B')],
 2)

In [22]:
# Test on test2.fa
print("=== test2.fa ===")
run_viterbi('example.hmm', 'test2.fa')

=== test2.fa ===
Segment Output:
1 26 state A
27 71 state B
72 120 state A

Number of segments in state B: 1


([(1, 26, 'state A'), (27, 71, 'state B'), (72, 120, 'state A')], 1)

In [23]:
# Test on test3.fa
print("=== test3.fa ===")
run_viterbi('example.hmm', 'test3.fa')

=== test3.fa ===
Segment Output:
1 38 state B
39 114 state A
115 152 state B
153 190 state A

Number of segments in state B: 2


([(1, 38, 'state B'),
  (39, 114, 'state A'),
  (115, 152, 'state B'),
  (153, 190, 'state A')],
 2)

## Summary

**Key Implementation Details:**
- Log probabilities prevent numerical underflow for long sequences
- Backpointer table stores only the previous state (memory efficient)